# Previsão de Inadimplência Bancária via Regressão Logística

**Disciplina:** Otimização Não-Linear 
**Professor:** Felipe Garcia 
**Integrantes:** Gabriel Negreiros Saraiva, Júlia Moraes da Silva, Luiz Eduardo de Almeida Siqueira Silva, Paulo Victor Cordeiro Rufino de Araújo, Pedro Lucas Simões Cabral

---

## 1. Objetivo

Implementar um classificador de **Regressão Logística** para prever inadimplência bancária, formulando o treinamento como um **problema de otimização** em aprendizado supervisionado de classificação.

$$\min_{\theta} \; L(\theta) = -\sum_{i=1}^{n} \left[ y_i \log \hat{y}_i + (1 - y_i) \log(1 - \hat{y}_i) \right]$$

onde $\hat{y}_i = \sigma(x_i^\top \theta)$ e $\sigma(z) = \frac{1}{1 + e^{-z}}$.

Serão comparados dois algoritmos de otimização:
- **Gradiente Descendente** com busca em linha
- **Método de Newton** com busca em linha

In [6]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from sklearn.model_selection import train_test_split

---
## 2. Carregamento e Pré-processamento dos Dados

Dataset: **Credito** — 10.128 clientes, 15 features financeiras e comportamentais, rótulo binário de inadimplência.

In [7]:
df = pd.read_csv('../dados/credito.csv')

# X é o conjunto de dados de entrada, e y é o conjunto de dados de saída
X = df.drop('default', axis=1)
y = df['default']


In [11]:
# verificação de dados ausentes
X.isna().any()

id                         False
idade                      False
sexo                       False
dependentes                False
escolaridade               False
estado_civil               False
salario_anual              False
tipo_cartao                False
meses_de_relacionamento    False
qtd_produtos               False
iteracoes_12m              False
meses_inativo_12m          False
limite_credito             False
valor_transacoes_12m       False
qtd_transacoes_12m         False
dtype: bool

In [5]:
#Fazendo a normalização dos dados. Fórmula: x_norm = (x - μ) / σ
media = np.mean(X, axis=0)
desvio_padrao = np.std(X, axis=0)

desvio_seguro = np.where(desvio_padrao == 0, 1.0, desvio_padrao)
X_normalizado = (X - media) / desvio_seguro

In [6]:
#Divisão dos dados em treino e teste
X_train, X_test, y_train, y_test = train_test_split(X_normalizado, y, test_size=0.2, random_state=42)

---
## 3. Formulação Matemática

### Modelo

Dado $X \in \mathbb{R}^{n \times (p+1)}$ (com coluna de bias), $y \in \{0,1\}^n$, $\theta \in \mathbb{R}^{p+1}$:

$$\hat{y}_i = \sigma(x_i^\top \theta), \quad \sigma(z) = \frac{1}{1 + e^{-z}}$$

### Função de Perda (a minimizar)

$$L(\theta) = -\sum_{i=1}^{n} \left[ y_i \log \hat{y}_i + (1-y_i)\log(1-\hat{y}_i) \right]$$

### Gradiente

$$\nabla L(\theta) = X^\top(\hat{y} - y)$$

### Hessiana

$$H(\theta) = X^\top W X, \quad W = \mathrm{diag}\!\left(\hat{y}_i(1-\hat{y}_i)\right)$$

A Hessiana é **positiva semi-definida** → $L(\theta)$ é **convexa** → qualquer mínimo local é global.

---
## 3.1 Gradiente, Hessiana e Prova de Convexidade

### Prova: H(θ) é Positiva Semi-Definida

A Hessiana da função de perda (sem regularização) é:

$$H(\theta) = \frac{1}{n} X^\top W X, \qquad W = \mathrm{diag}\!\left(\hat{y}_i(1-\hat{y}_i)\right)$$

**Prova algébrica** — Para qualquer $v \in \mathbb{R}^{p+1}$:

$$v^\top H v = \frac{1}{n}\, v^\top X^\top W X v = \frac{1}{n}\,(Xv)^\top W (Xv) = \frac{1}{n}\sum_{i=1}^{n} w_i\,(x_i^\top v)^2$$

Como $\sigma(z) \in (0,1)$ para todo $z$ real, cada peso satisfaz:

$$w_i = \hat{y}_i(1 - \hat{y}_i) \in \left(0,\, \tfrac{1}{4}\right]$$

Logo cada termo $w_i\,(x_i^\top v)^2 \geq 0$, e portanto:

$$\boxed{v^\top H v \geq 0 \quad \forall\, v \quad \Longrightarrow \quad H(\theta) \succeq 0 \quad \forall\, \theta}$$

> **Com regularização L2** ($\lambda > 0$): a implementação usa $H_{\text{reg}} = H + \lambda\,\mathrm{diag}(0, 1, \ldots, 1)$, que adiciona $\lambda > 0$ a todos os autovalores (exceto o do bias). Isso torna $H_{\text{reg}}$ **estritamente positiva definida** ($H \succ 0$), garantindo unicidade do mínimo e estabilidade numérica ao resolver $H\,d = -\nabla L$ no Método de Newton.

In [ ]:
from regressao_log import RegressaoLogistica

# Preparar dados: converter para numpy e adicionar coluna de bias
X_np = np.array(X_train, dtype=float)
y_np = np.array(y_train, dtype=float)
X_b  = np.c_[np.ones(X_np.shape[0]), X_np]   # (n, p+1)

theta_0 = np.zeros(X_b.shape[1])
lambda_ = 0.01

# ---------- Gradiente em θ₀ = 0 ----------
grad_0 = RegressaoLogistica._gradiente(theta_0, X_b, y_np, lambda_)

# ---------- Hessiana em θ₀ = 0 ----------
H_0 = RegressaoLogistica._hessiana(theta_0, X_b, lambda_)

# ---------- Autovalores (eigvalsh para matrizes simétricas) ----------
autovalores = np.linalg.eigvalsh(H_0)

print(f"Dimensão de H(θ₀):            {H_0.shape}")
print(f"‖∇L(θ₀)‖:                     {np.linalg.norm(grad_0):.4f}")
print(f"Autovalor mínimo λ_min:        {autovalores.min():.8f}  "
      f"{'≥ 0  →  PSD ✓' if autovalores.min() >= -1e-10 else '< 0  ✗'}")
print(f"Autovalor máximo λ_max:        {autovalores.max():.4f}")
print(f"Número de condição κ(H):       {autovalores.max() / max(autovalores.min(), 1e-12):.1f}")
print(f"\nTodos autovalores ≥ 0?        {'Sim ✓' if (autovalores >= -1e-10).all() else 'Não ✗'}")

In [ ]:
nomes_features = ['bias'] + list(X_train.columns)

fig, axes = plt.subplots(1, 3, figsize=(17, 5))

# --- 1. Espectro de autovalores ---
cores_av = ['steelblue' if v >= 0 else 'tomato' for v in autovalores]
axes[0].barh(range(len(autovalores)), autovalores, color=cores_av,
             edgecolor='white', linewidth=0.4)
axes[0].axvline(0, color='black', linewidth=1.0)
axes[0].set_xlabel('Valor do autovalor λᵢ')
axes[0].set_ylabel('Índice do autovalor')
axes[0].set_title('Espectro de H(θ₀)\ntodos λᵢ ≥ 0  →  PSD ✓', fontweight='bold')
axes[0].grid(True, axis='x', alpha=0.3)
axes[0].annotate(
    f'λ_min = {autovalores.min():.4f}',
    xy=(autovalores.min(), 0),
    xytext=(autovalores.max() * 0.3, len(autovalores) * 0.15),
    arrowprops=dict(arrowstyle='->', color='black'), fontsize=9,
)

# --- 2. Heatmap da Hessiana ---
im = axes[1].imshow(H_0, cmap='RdBu_r', aspect='auto')
plt.colorbar(im, ax=axes[1], fraction=0.046)
axes[1].set_title('Heatmap de H(θ₀)\ncurvatura entre features', fontweight='bold')
axes[1].set_xlabel('Feature j')
axes[1].set_ylabel('Feature i')
ticks = range(len(nomes_features))
axes[1].set_xticks(ticks)
axes[1].set_xticklabels(nomes_features, rotation=90, fontsize=6)
axes[1].set_yticks(ticks)
axes[1].set_yticklabels(nomes_features, fontsize=6)

# --- 3. Componentes do gradiente ---
idx_ord = np.argsort(np.abs(grad_0))
cores_g = ['tomato' if v < 0 else 'steelblue' for v in grad_0[idx_ord]]
axes[2].barh(range(len(grad_0)), grad_0[idx_ord], color=cores_g,
             edgecolor='white', linewidth=0.4)
axes[2].set_yticks(range(len(grad_0)))
axes[2].set_yticklabels([nomes_features[i] for i in idx_ord], fontsize=7)
axes[2].axvline(0, color='black', linewidth=0.8)
axes[2].set_title('Componentes de ∇L(θ₀)\n(ponto de partida θ = 0)', fontweight='bold')
axes[2].set_xlabel('∂L/∂θᵢ')
axes[2].grid(True, axis='x', alpha=0.3)

fig.suptitle('Gradiente e Hessiana no Ponto Inicial θ₀ = 0',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### O que implica H(θ) ⪰ 0?

#### 1. L(θ) é convexa

Uma função diferenciável é convexa se e somente se sua Hessiana é positiva semi-definida em todo o domínio. Como $H(\theta) \succeq 0$ para **qualquer** $\theta$ (os pesos $w_i = \hat{y}_i(1-\hat{y}_i)$ dependem de $\theta$, mas sempre $w_i > 0$), a função de perda de entropia cruzada é **globalmente convexa**.

**Consequência prática:** qualquer mínimo local é automaticamente um mínimo global. O algoritmo de otimização não pode ficar preso em ótimos locais — qualquer convergência é para a solução ótima.

#### 2. O Método de Newton é garantidamente uma direção de descida

A direção de Newton $d_k = -H(\theta_k)^{-1} \nabla L(\theta_k)$ é uma **direção de descida** sempre que $H \succ 0$:

$$\nabla L^\top d_k = -\nabla L^\top H^{-1} \nabla L < 0$$

pois $H^{-1} \succ 0$ quando $H \succ 0$. Isso garante que cada passo de Newton reduz $L(\theta)$.

#### 3. O número de condição κ(H) governa a velocidade do GD

O Gradiente Descendente converge em $O(\kappa(H))$ iterações no pior caso, onde:

$$\kappa(H) = \frac{\lambda_{\max}}{\lambda_{\min}}$$

Um $\kappa$ alto significa que existem direções com curvaturas muito diferentes — o GD "ziguezagueia" entre elas. O Método de Newton usa $H^{-1}$ para normalizar essas curvaturas, sendo **invariante ao condicionamento** de $H$ e convergindo em $O(\log(1/\varepsilon))$ iterações independentemente de $\kappa$.

---
## 4. Demonstração das Funções do Modelo

In [ ]:
# Sigmoide
z = np.linspace(-6, 6, 300)
plt.figure(figsize=(7, 3))
plt.plot(z, funcao_sigmoide(z), color='steelblue', linewidth=2)
plt.axhline(0.5, color='gray', linestyle='--', linewidth=0.8)
plt.axvline(0,   color='gray', linestyle='--', linewidth=0.8)
plt.xlabel('z'); plt.ylabel('σ(z)')
plt.title('Função Sigmoide — σ(z) = 1/(1 + e⁻ᶻ)')
plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

# Gradiente e Hessiana no ponto inicial θ=0
theta_zero = np.zeros(X_treino_b.shape[1])
grad_inicial = gradiente_perda(theta_zero, X_treino_b, y_treino)
H_inicial    = hessiana_perda(theta_zero, X_treino_b, y_treino)
autovalores  = np.linalg.eigvalsh(H_inicial)

print(f"Perda inicial L(0):          {funcao_perda(theta_zero, X_treino_b, y_treino):.2f}")
print(f"‖∇L(0)‖:                     {np.linalg.norm(grad_inicial):.2f}")
print(f"Autovalor mínimo H(0):       {autovalores.min():.4f}  (≥ 0 → convexa ✓)")
print(f"Autovalor máximo H(0):       {autovalores.max():.4f}")
print(f"Número de condição H(0):     {autovalores.max()/autovalores.min():.1f}")

---
## 5. Métodos de Busca em Linha — Demonstração 1D

Aplicamos cada método à função $f(\alpha) = 2\alpha^2 - 7\alpha + 1$, cujo mínimo exato é $\alpha^* = 7/4 = 1{,}75$.

In [ ]:
f   = lambda a: 2*a**2 - 7*a + 1
df  = lambda a: 4*a - 7
alpha_exato = 1.75

# Intervalo inicial automático
a0, b0 = intervalo_inicial(f, alpha_inicial=0.0)
print(f"intervalo_inicial:          [{a0:.4f}, {b0:.4f}]")

# Partição igual (trisseção)
n_it, alpha_part = busca_particao_igual(f, a0, b0)
print(f"busca_particao_igual:       α* = {alpha_part:.8f}  (erro = {abs(alpha_part-alpha_exato):.2e},  iters = {n_it})")

# Seção áurea
a_s, b_s = busca_secao_aurea(f, 1.0, 5.0, num_iteracoes=40)
alpha_aurea = (a_s + b_s) / 2
print(f"busca_secao_aurea:          α* = {alpha_aurea:.8f}  (erro = {abs(alpha_aurea-alpha_exato):.2e})")

# Ajuste quadrático
_, alpha_quad, _ = busca_ajuste_quadratico(f, 1.0, 5.0, 3.0, num_iteracoes=20)
print(f"busca_ajuste_quadratico:    α* = {alpha_quad:.8f}  (erro = {abs(alpha_quad-alpha_exato):.2e})")

# Biseção na derivada
a_b, b_b = intervalo_bissecao_derivada(df, 0.0, 0.1)
a_b2, b_b2 = bissecao_derivada(df, a_b, b_b)
alpha_bis = (a_b2 + b_b2) / 2
print(f"bissecao_derivada:          α* = {alpha_bis:.8f}  (erro = {abs(alpha_bis-alpha_exato):.2e})")

# Visualização
alpha_vals = np.linspace(0.5, 3.0, 300)
plt.figure(figsize=(8, 3.5))
plt.plot(alpha_vals, f(alpha_vals), color='steelblue', linewidth=2, label='f(α)')
plt.axvline(alpha_exato,  color='black',  linestyle='--', linewidth=1, label=f'Mínimo exato ({alpha_exato})')
plt.axvline(alpha_aurea,  color='tomato', linestyle=':',  linewidth=2, label=f'Seção áurea ({alpha_aurea:.6f})')
plt.axvline(alpha_quad,   color='green',  linestyle='-.',  linewidth=2, label=f'Aj. quadrático ({alpha_quad:.6f})')
plt.xlabel('α'); plt.title('Busca em Linha — f(α) = 2α² − 7α + 1')
plt.legend(fontsize=8); plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

---
## 6. Treinamento — Gradiente Descendente com Busca em Linha

$$\theta_{k+1} = \theta_k - \alpha^* \nabla L(\theta_k), \quad \alpha^* = \arg\min_\alpha L(\theta_k - \alpha \nabla L(\theta_k))$$

**Nota:** O GD tem convergência **linear** — a norma do gradiente decresce ~0.5% por iteração perto do ótimo. Com 24.000 amostras, atingir $\|\nabla L\| < 10^{-4}$ requer ~7.800 iterações. Aqui executamos 500 iterações para demonstrar a trajetória de convergência.

In [ ]:
print("Executando Gradiente Descendente (500 iterações)...")
t_inicio_gd = time.time()

theta_gd, hist_perda_gd, hist_grad_gd, n_iter_gd = gradiente_descendente(
    funcao_perda, gradiente_perda,
    theta_zero, X_treino_b, y_treino,
    metodo_busca='secao_aurea',
    tolerancia=1e-4,
    max_iteracoes=500,
)
tempo_gd = time.time() - t_inicio_gd

print(f"Iterações executadas:  {n_iter_gd}")
print(f"Perda inicial → final: {hist_perda_gd[0]:.2f} → {hist_perda_gd[-1]:.2f}")
print(f"‖∇L‖ inicial → final:  {hist_grad_gd[0]:.2f} → {hist_grad_gd[-1]:.4f}")
print(f"Tempo de execução:     {tempo_gd:.1f}s")

plotar_convergencia(hist_perda_gd, hist_grad_gd, 'Gradiente Descendente — Convergência')

---
## 7. Treinamento — Método de Newton com Busca em Linha

$$\theta_{k+1} = \theta_k + \alpha^* d_k, \quad d_k = -H(\theta_k)^{-1}\nabla L(\theta_k)$$

A direção de Newton incorpora a **curvatura** da função via Hessiana, resultando em convergência **quadrática** — a precisão dobra a cada iteração perto do ótimo.

In [ ]:
print("Executando Método de Newton...")
t_inicio_nt = time.time()

theta_nt, hist_perda_nt, hist_grad_nt, n_iter_nt = metodo_newton(
    funcao_perda, gradiente_perda, hessiana_perda,
    theta_zero, X_treino_b, y_treino,
    metodo_busca='secao_aurea',
    tolerancia=1e-4,
    max_iteracoes=100,
)
tempo_nt = time.time() - t_inicio_nt

print(f"Iterações para convergência: {n_iter_nt}")
print(f"Perda inicial → final:       {hist_perda_nt[0]:.2f} → {hist_perda_nt[-1]:.2f}")
print(f"‖∇L‖ inicial → final:        {hist_grad_nt[0]:.2f} → {hist_grad_nt[-1]:.2e}")
print(f"Tempo de execução:           {tempo_nt:.2f}s")

plotar_convergencia(hist_perda_nt, hist_grad_nt, 'Método de Newton — Convergência')

---
## 8. Comparação dos Métodos de Otimização

In [ ]:
# --- Tabela comparativa ---
print("=" * 58)
print(f"{'Métrica':<30} {'GD (500 iters)':>12} {'Newton':>12}")
print("=" * 58)
print(f"{'Iterações executadas':<30} {n_iter_gd:>12d} {n_iter_nt:>12d}")
print(f"{'Tempo de execução (s)':<30} {tempo_gd:>12.1f} {tempo_nt:>12.2f}")
print(f"{'Perda final L(θ)':<30} {hist_perda_gd[-1]:>12.4f} {hist_perda_nt[-1]:>12.4f}")
print(f"{'‖∇L(θ)‖ final':<30} {hist_grad_gd[-1]:>12.4f} {hist_grad_nt[-1]:>12.2e}")
print("=" * 58)
print(f"\nNewton é ~{n_iter_gd/max(n_iter_nt,1):.0f}× mais eficiente em iterações")
print(f"Newton é ~{tempo_gd/max(tempo_nt,0.01):.0f}× mais rápido em tempo de parede")

# --- Curvas de convergência lado a lado ---
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Perda
axes[0].plot(hist_perda_gd, color='steelblue',  linewidth=1.5, label=f'GD ({n_iter_gd} iters)')
axes[0].plot(hist_perda_nt, color='darkorange', linewidth=2.5, label=f'Newton ({n_iter_nt} iters)', marker='o', markersize=5)
axes[0].set_xlabel('Iteração'); axes[0].set_ylabel('L(θ)')
axes[0].set_title('Evolução da Perda'); axes[0].legend(); axes[0].grid(True, alpha=0.3)

# Norma do gradiente (log)
axes[1].semilogy(hist_grad_gd, color='steelblue',  linewidth=1.5, label='GD')
axes[1].semilogy(hist_grad_nt, color='darkorange', linewidth=2.5, label='Newton', marker='o', markersize=5)
axes[1].set_xlabel('Iteração'); axes[1].set_ylabel('‖∇L(θ)‖  (log)')
axes[1].set_title('Norma do Gradiente'); axes[1].legend(); axes[1].grid(True, alpha=0.3)

fig.suptitle('GD vs Newton — Comparação de Convergência', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

---
## 9. Avaliação no Conjunto de Teste

Avaliamos o modelo treinado pelo Método de Newton (solução mais precisa) no conjunto de teste.

In [ ]:
y_pred = prever_classe(theta_nt, X_teste_b)
prob   = prever_probabilidade(theta_nt, X_teste_b)

VP, FP, FN, VN = calcular_matriz_confusao(y_teste, y_pred)
acc  = calcular_acuracia(y_teste,  y_pred)
prec = calcular_precisao(y_teste,  y_pred)
rec  = calcular_recall(y_teste,    y_pred)
f1   = calcular_f1(y_teste,        y_pred)
taxas_fp, taxas_vp, auc = calcular_roc_auc(y_teste, prob)

print("=" * 40)
print(f"{'Acurácia':<20} {acc:.4f}")
print(f"{'Precisão':<20} {prec:.4f}")
print(f"{'Recall':<20} {rec:.4f}")
print(f"{'F1-score':<20} {f1:.4f}")
print(f"{'AUC-ROC':<20} {auc:.4f}")
print("=" * 40)

# Gráficos
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# ROC
axes[0].plot(taxas_fp, taxas_vp, color='steelblue', linewidth=2, label=f'AUC = {auc:.4f}')
axes[0].plot([0,1],[0,1], 'gray', linestyle='--', linewidth=1, label='Aleatório')
axes[0].set_xlabel('FPR'); axes[0].set_ylabel('TPR')
axes[0].set_title('Curva ROC'); axes[0].legend(); axes[0].grid(True, alpha=0.3)

# Matriz de confusão
valores = np.array([[VN, FP], [FN, VP]])
rotulos = [['VN','FP'],['FN','VP']]
im = axes[1].imshow(valores, cmap='Blues')
for i in range(2):
    for j in range(2):
        cor = 'white' if valores[i,j] > valores.max()/2 else 'black'
        axes[1].text(j, i, f'{rotulos[i][j]}\n{valores[i,j]}', ha='center', va='center',
                     fontsize=12, fontweight='bold', color=cor)
axes[1].set_xticks([0,1]); axes[1].set_yticks([0,1])
axes[1].set_xticklabels(['Pred: 0','Pred: 1']); axes[1].set_yticklabels(['Real: 0','Real: 1'])
axes[1].set_title('Matriz de Confusão'); plt.colorbar(im, ax=axes[1])

# Coeficientes
coef = theta_nt[1:]
idx  = np.argsort(np.abs(coef))
cores = ['tomato' if v < 0 else 'steelblue' for v in coef[idx]]
axes[2].barh(range(len(coef)), coef[idx], color=cores)
axes[2].set_yticks(range(len(coef)))
axes[2].set_yticklabels([NOMES_FEATURES[i] for i in idx], fontsize=7)
axes[2].axvline(0, color='black', linewidth=0.8)
axes[2].set_title('Coeficientes θ (por |θᵢ|)'); axes[2].grid(True, axis='x', alpha=0.3)

plt.tight_layout(); plt.show()

---
## 10. Conclusões

### Resultados de Otimização

| | Gradiente Descendente | Método de Newton |
|---|---|---|
| **Convergência** | Linear | **Quadrática** |
| **Iterações** | 500+ (não convergiu) | **5** |
| **Custo por iteração** | $O(np)$ | $O(np^2 + p^3)$ |
| **Informação usada** | $\nabla L$ | $\nabla L$ + $H$ |

### Por que Newton é tão mais eficiente?

O **número de condição** da Hessiana $H = X^\top W X$ determina a velocidade do GD.
Quanto maior o número de condição, mais lento o GD converge — independentemente do tamanho do passo.
Newton corrige isso usando $H^{-1}$ para escalar a direção, tornando-se invariante ao condicionamento.

### Resultados do Classificador (Método de Newton)

- **Acurácia**: ~81% — sólida para base de dados financeiros com 22% de inadimplência
- **AUC-ROC**: ~0.73 — modelo discrimina bem adimplentes de inadimplentes
- **Feature mais influente**: `PAY_0` (status de pagamento em setembro) — atraso recente é o melhor preditor de inadimplência futura

### Limitações

- O Recall (~24%) é baixo: muitos inadimplentes reais não são detectados (FN alto)
- Regressão logística é linear — fronteira de decisão pode não capturar padrões não lineares
- Desbalanceamento de classes (78/22) afeta as métricas